In [1]:
import sys
!{sys.executable} -m pip install pandas scikit-learn

In [2]:
import pandas as pd
print("Pandas OK")

Pandas OK


In [6]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import os
import re
import math
from collections import Counter
import pandas as pd

ruta_turismo = "/content/drive/MyDrive/data/01_corpus_turismo_500.txt"

with open(ruta_turismo, "r", encoding="utf-8", errors="ignore") as f:
    lineas = f.readlines()

corpus_turismo = [linea.strip() for linea in lineas if linea.strip()]

print("Cantidad de documentos turismo:", len(corpus_turismo))
print(corpus_turismo[:3])

Cantidad de documentos turismo: 500
['Otavalo es conocido por su mercado indígena y su artesanía Perfecto para rafting.', 'La Laguna Quilotoa destaca por su color turquesa', 'Vilcabamba atrae visitantes interesados en longevidad y naturaleza Una experiencia inolvidable.']


### Preprocesamiento del corpus de turismo

Como se indicó en el Paso 1, antes de calcular TF-IDF, se realizará un preprocesamiento básico en el `corpus_turismo`:
- Conversión a minúsculas
- Eliminación de caracteres especiales
- Tokenización del texto

In [8]:
def tokenizar(texto):
    # Convierte a minúsculas
    texto = texto.lower()
    # Elimina caracteres especiales y deja solo letras, números y espacios
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)
    # Divide el texto en tokens (palabras)
    return texto.split()

# Aplicar la tokenización al corpus de turismo
corpus_turismo_tokenizado = [tokenizar(doc) for doc in corpus_turismo]

print("Primer documento tokenizado:")
print(corpus_turismo_tokenizado[0])

Primer documento tokenizado:
['otavalo', 'es', 'conocido', 'por', 'su', 'mercado', 'indígena', 'y', 'su', 'artesanía', 'perfecto', 'para', 'rafting']


### Cálculo de la matriz TF-IDF para el corpus de turismo

Ahora, utilizaremos `TfidfVectorizer` de `sklearn` para calcular la matriz TF-IDF. Esto convertirá nuestros documentos tokenizados en vectores numéricos, donde cada valor representa la importancia de un término en un documento específico.

In [15]:
carpeta_gutenberg = "/content/drive/MyDrive/data/corpus_limpio/corpus_limpio"

archivos_gutenberg = sorted([
    archivo for archivo in os.listdir(carpeta_gutenberg)
    if archivo.endswith(".txt")
])

corpus_gutenberg = []
nombres_documentos = []

for archivo in archivos_gutenberg:
    ruta = os.path.join(carpeta_gutenberg, archivo)

    with open(ruta, "r", encoding="utf-8", errors="ignore") as f:
        texto = f.read()

    if texto.strip():
        corpus_gutenberg.append(texto)
        nombres_documentos.append(archivo)

print("Cantidad de libros cargados:", len(corpus_gutenberg))
print("Primeros archivos:", nombres_documentos[:10])

Cantidad de libros cargados: 1000
Primeros archivos: ['10084-8.txt', '10084.txt', '1032.txt', '1037.txt', '1093.txt', '1096.txt', '1143.txt', '1149.txt', '1160.txt', '1161.txt']


In [13]:
import os

data_path = '/content/drive/MyDrive/data'
if os.path.exists(data_path):
    print(f"Contenido de {data_path}:")
    for item in os.listdir(data_path):
        print(item)
else:
    print(f"La carpeta {data_path} no existe. Por favor, verifica que tu Google Drive esté montado correctamente y que la ruta sea la indicada.")


Contenido de /content/drive/MyDrive/data:
01_corpus_turismo_500.txt
corpus_limpio.zip


In [14]:
import zipfile
import os

zip_path = '/content/drive/MyDrive/data/corpus_limpio.zip'
extract_path = '/content/drive/MyDrive/data/corpus_limpio'

# Crear el directorio de destino si no existe
os.makedirs(extract_path, exist_ok=True)

print(f"Descomprimiendo {zip_path} en {extract_path}...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print("Descompresión completada.")

# Verificar el contenido de la nueva carpeta (opcional)
print(f"Contenido de {extract_path}:")
print(os.listdir(extract_path)[:5]) # Mostrar los primeros 5 archivos para verificar

Descomprimiendo /content/drive/MyDrive/data/corpus_limpio.zip en /content/drive/MyDrive/data/corpus_limpio...
Descompresión completada.
Contenido de /content/drive/MyDrive/data/corpus_limpio:
['corpus_limpio']


In [16]:

from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

# Descargar las stopwords si aún no están descargadas
try:
    stopwords.words('english') # Assuming Gutenberg is primarily English for stopwords
except LookupError:
    nltk.download('stopwords')

english_stopwords = stopwords.words('english')

# Preprocesar el corpus Gutenberg utilizando la misma función de tokenización
# Aseguramos que 'tokenizar' esté definida, ya lo está para el corpus de turismo
corpus_gutenberg_tokenizado = [tokenizar(doc) for doc in corpus_gutenberg]
corpus_gutenberg_para_tfidf = [" ".join(doc) for doc in corpus_gutenberg_tokenizado]

# Crear vectorizador TF-IDF para Gutenberg
tfidf_vectorizer_gutenberg = TfidfVectorizer(
    max_features=5000,
    stop_words=english_stopwords
)

# Calcular TF-IDF
tfidf_matrix_gutenberg = tfidf_vectorizer_gutenberg.fit_transform(corpus_gutenberg_para_tfidf)

# Convertir a DataFrame
tfidf_gutenberg = pd.DataFrame(
    tfidf_matrix_gutenberg.toarray(),
    columns=tfidf_vectorizer_gutenberg.get_feature_names_out()
)

# Mostrar resultados
print("Tamaño matriz TF-IDF Gutenberg:")
print(tfidf_gutenberg.shape)

tfidf_gutenberg.head()

Tamaño matriz TF-IDF Gutenberg:
(1000, 5000)


,abandon,abandoned,abbe,ability,able,aboard,abode,abroad,abruptly,absence,...,yield,yielded,yonder,york,young,younger,youth,youthful,zeal,zeus
0,0.000000,0.000311,0.0,0.000000,0.000823,0.0,0.000000,0.000280,0.000000,0.000256,...,0.000000,0.000000,0.0,0.000000,0.004548,0.002299,0.000439,0.000000,0.000000,0.0
1,0.000000,0.000311,0.0,0.000000,0.000823,0.0,0.000000,0.000280,0.000000,0.000256,...,0.000000,0.000000,0.0,0.000000,0.004548,0.002299,0.000439,0.000000,0.000000,0.0
2,0.004629,0.000000,0.0,0.004173,0.015441,0.0,0.000000,0.003506,0.004276,0.003205,...,0.000000,0.000000,0.0,0.004055,0.099545,0.003194,0.027461,0.000000,0.000000,0.0
3,0.000000,0.000000,0.0,0.000940,0.009270,0.0,0.001093,0.002368,0.000000,0.001443,...,0.003508,0.001901,0.0,0.000913,0.012806,0.000719,0.001236,0.001943,0.005233,0.0
4,0.004645,0.003899,0.0,0.000000,0.023240,0.0,0.000000,0.000000,0.000000,0.019293,...,0.000000,0.000000,0.0,0.000000,0.011891,0.003205,0.013777,0.000000,0.000000,0.0


In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
from nltk.corpus import stopwords

# Descargar las stopwords si aún no están descargadas
try:
    stopwords.words('spanish')
except LookupError:
    nltk.download('stopwords')

spanish_stopwords = stopwords.words('spanish')

# Unir los tokens para que TfidfVectorizer pueda procesarlos como cadenas de texto
corpus_turismo_para_tfidf = [" ".join(doc) for doc in corpus_turismo_tokenizado]

# Crear el vectorizador TF-IDF
# max_features limita el número de términos a considerar, stop_words elimina palabras comunes
tfidf_vectorizer_turismo = TfidfVectorizer(
    max_features=1000, # Limitar a 1000 características para este corpus pequeño
    stop_words=spanish_stopwords # Usar la lista de stopwords en español de NLTK
)

# Calcular la matriz TF-IDF
tfidf_matrix_turismo = tfidf_vectorizer_turismo.fit_transform(corpus_turismo_para_tfidf)

# Convertir a DataFrame para una mejor visualización
tfidf_turismo_df = pd.DataFrame(
    tfidf_matrix_turismo.toarray(),
    columns=tfidf_vectorizer_turismo.get_feature_names_out()
)

print("Tamaño de la matriz TF-IDF para el corpus de turismo:")
print(tfidf_turismo_df.shape)

print("Primeras 5 filas de la matriz TF-IDF de turismo:")
display(tfidf_turismo_df.head())

Tamaño de la matriz TF-IDF para el corpus de turismo:
(500, 95)
Primeras 5 filas de la matriz TF-IDF de turismo:


,agua,amazonía,arquitectura,artesanía,atrae,atraen,auténtico,aventura,aves,avistamiento,...,turistas,turquesa,típica,vida,vilcabamba,visitan,visitantes,visitar,volcán,única
0,0.0,0.0,0.0,0.399895,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
1,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.447214,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
2,0.0,0.0,0.0,0.000000,0.390303,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.390303,0.0,0.312063,0.0,0.0,0.0
3,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0
4,0.0,0.0,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,...,0.0,0.000000,0.0,0.0,0.000000,0.0,0.000000,0.0,0.0,0.0


### Demostración de la función `buscar()`

La función `buscar()` ya está disponible y lista para ser utilizada con nuevas consultas.

In [19]:
import re
import pandas as pd # Assuming pandas is needed for DataFrame construction

def tokenizar(texto):
    # Convierte a minúsculas
    texto = texto.lower()
    # Elimina caracteres especiales y deja solo letras, números y espacios
    texto = re.sub(r"[^a-záéíóúñü\s]", " ", texto)
    # Divide el texto en tokens (palabras)
    return texto.split()

def buscar(consulta, top_n=10):
    tokens_consulta = tokenizar(consulta)

    resultados = []

    for i in range(len(corpus_gutenberg)):
        score = 0

        for termino in tokens_consulta:
            if termino in tfidf_gutenberg.columns:
                score += tfidf_gutenberg.iloc[i][termino]

        resultados.append((nombres_documentos[i], score))

    resultados = sorted(resultados, key=lambda x: x[1], reverse=True)

    return pd.DataFrame(resultados[:top_n], columns=["Documento", "Score TF-IDF"])

In [20]:
# Puedes probar la función 'buscar' con una nueva consulta
consulta_ejemplo = "war politics revolution"
resultados_nueva_consulta = buscar(consulta_ejemplo, top_n=5)
display(resultados_nueva_consulta)

,Documento,Score TF-IDF
0,1804.txt,0.414321
1,1946.txt,0.344842
2,1946-8.txt,0.344840
3,1864.txt,0.235072
4,1484.txt,0.207470
